In [0]:
-- Set the default catalog and schema for this notebook.
-- All unqualified table references will resolve to market_risk_dev.silver
USE CATALOG market_risk_dev;
USE SCHEMA silver;

In [0]:
-- ─────────────────────────────────────────────────────────────────────────────
-- silver.prices_cleaned
-- Purpose : Deduplicated, null-free daily market prices
-- Source  : market_risk_dev.bronze.market_prices
-- Grain   : One row per ticker per trading day
-- Notes   : Removes weekends, holidays and any duplicate loads.
--           ZORDER candidate: (Date, ticker) — add once query patterns confirmed
-- ─────────────────────────────────────────────────────────────────────────────
CREATE OR REPLACE TABLE market_risk_dev.silver.prices_cleaned (
  price_date        DATE      NOT NULL COMMENT 'Trading date — weekends and holidays excluded',
  open_price        DOUBLE    NOT NULL COMMENT 'Opening price in local currency',
  high_price        DOUBLE    NOT NULL COMMENT 'Intraday high price in local currency',
  low_price         DOUBLE    NOT NULL COMMENT 'Intraday low price in local currency',
  close_price       DOUBLE    NOT NULL COMMENT 'Closing price in local currency — primary price used in calculations',
  volume            DOUBLE             COMMENT 'Volume traded — null for FX pairs which report no volume',
  ticker            STRING    NOT NULL COMMENT 'Yahoo Finance ticker symbol e.g. AAPL, EURUSD=X',
  fetched_at        STRING             COMMENT 'Timestamp when data was fetched from Yahoo Finance API',
  _ingested_at      TIMESTAMP          COMMENT 'Timestamp when row was loaded into Bronze',
  _silver_loaded_at TIMESTAMP          COMMENT 'Timestamp when row was loaded into Silver'
)
USING DELTA
COMMENT 'Deduplicated and cleaned market prices. One row per ticker per trading day. Source: bronze.market_prices.'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
);

In [0]:
-- ─────────────────────────────────────────────────────────────────────────────
-- Load silver.prices_cleaned
-- Strategy : INSERT OVERWRITE — full refresh each run.
--            Appropriate here because price history is small (~2,800 rows)
--            and corrections to historical prices should propagate fully.
--            For larger price histories (millions of rows) switch to MERGE.
-- ─────────────────────────────────────────────────────────────────────────────
INSERT OVERWRITE market_risk_dev.silver.prices_cleaned

WITH

-- Step 1: Remove rows with no usable price data.
--         Yahoo Finance returns null Close for weekends and some holidays.
--         We filter these out entirely rather than forward-filling because
--         forward-filled prices would produce misleading VaR calculations.
valid_prices AS (
  SELECT
    CAST(Date       AS DATE)      AS price_date,
    CAST(Open       AS DOUBLE)    AS open_price,
    CAST(High       AS DOUBLE)    AS high_price,
    CAST(Low        AS DOUBLE)    AS low_price,
    CAST(Close      AS DOUBLE)    AS close_price,
    CAST(Volume     AS DOUBLE)    AS volume,
    CAST(ticker     AS STRING)    AS ticker,
    CAST(fetched_at AS STRING)    AS fetched_at,
    _ingested_at
  FROM market_risk_dev.bronze.market_prices
  WHERE
    Date        IS NOT NULL
    AND Close   IS NOT NULL
    AND Open    IS NOT NULL
    AND High    IS NOT NULL
    AND Low     IS NOT NULL
    AND ticker  IS NOT NULL
),

-- Step 2: Deduplicate — keep only the most recently ingested record
--         for each ticker + date combination.
--         This handles the case where the same file is uploaded to S3
--         and re-ingested, which would otherwise create duplicate rows.
deduplicated AS (
  SELECT
    price_date,
    open_price,
    high_price,
    low_price,
    close_price,
    volume,
    ticker,
    fetched_at,
    _ingested_at,
    ROW_NUMBER() OVER (
      PARTITION BY ticker, price_date
      ORDER BY _ingested_at DESC
    ) AS row_num
  FROM valid_prices
)

-- Step 3: Keep only the canonical row per ticker per date
SELECT
  price_date,
  open_price,
  high_price,
  low_price,
  close_price,
  volume,
  ticker,
  fetched_at,
  _ingested_at,
  current_timestamp() AS _silver_loaded_at
FROM deduplicated
WHERE row_num = 1;

In [0]:
-- ─────────────────────────────────────────────────────────────────────────────
-- Data quality gate for silver.prices_cleaned
-- All quality checks should pass before proceeding to positions_enriched.
-- Any NULL count > 0 in critical columns indicates a pipeline problem.
-- ─────────────────────────────────────────────────────────────────────────────
WITH 
quality_summary AS (
  SELECT 
    count(*)                                                              AS total_rows, 
    count(DISTINCT ticker)                                                AS unique_tickers,
    count(DISTINCT price_date)                                            AS unique_dates,
    MIN(price_date)                                                       AS earliest_date,
    MAX(price_date)                                                       AS latest_date,
    -- Critical null checks    
    SUM(CASE WHEN price_date IS NULL THEN 1 END)                          AS null_price_date,
    SUM(CASE WHEN ticker IS NULL THEN 1 END)                              AS null_ticker,
    SUM(CASE WHEN close_price IS NULL THEN 1 END)                         AS null_close_price,
    -- Duplicate check — should always be zero after deduplication
    COUNT(*) - COUNT(DISTINCT CONCAT(CAST(price_date AS STRING), ticker)) AS duplicate_rows,
    -- Sanity check: close price should always be positive
    COUNT(CASE WHEN close_price <= 0 THEN 1 END)                          AS non_positive_close_price
  FROM market_risk_dev.silver.prices_cleaned
)
SELECT 
  total_rows, 
  unique_tickers,
  unique_dates,
  earliest_date,
  latest_date,
  null_price_date,
  null_ticker,
  null_close_price,
  duplicate_rows,
  non_positive_close_price,
  -- Overall pass/fail
  CASE 
    WHEN null_price_date > 0 THEN 'FAIL - null price_date found'
    WHEN null_ticker > 0 THEN 'FAIL - null ticker found'
    WHEN null_close_price > 0 THEN 'FAIL - null close_price found'
    WHEN duplicate_rows > 0 THEN 'FAIL - duplicate rows found'
    WHEN non_positive_close_price > 0 THEN 'FAIL - non-positive close_price found'
    ELSE 'PASS'
  END AS quality_status
FROM quality_summary;

In [0]:
-- ─────────────────────────────────────────────────────────────────────────────
-- silver.positions_enriched
-- Purpose : Positions joined to current market prices and FX rates,
--           all monetary values normalised to USD
-- Source  : bronze.positions + bronze.fx_rates + silver.prices_cleaned
-- Grain   : One row per trade_id
-- Notes   : ZORDER candidate: (desk, trade_date) — add once query patterns confirmed
-- ─────────────────────────────────────────────────────────────────────────────
CREATE OR REPLACE TABLE market_risk_dev.silver.positions_enriched (
  trade_id              STRING    NOT NULL COMMENT 'Unique trade identifier — primary key',
  desk                  STRING    NOT NULL COMMENT 'Trading desk that owns the position',
  counterparty          STRING    NOT NULL COMMENT 'Counterparty name',
  asset_class           STRING    NOT NULL COMMENT 'Asset class: FX, Equity, Rates, Credit',
  ticker                STRING    NOT NULL COMMENT 'Instrument ticker symbol',
  direction             STRING    NOT NULL COMMENT 'LONG or SHORT',
  notional              DOUBLE    NOT NULL COMMENT 'Notional value in local currency',
  currency              STRING    NOT NULL COMMENT 'Local currency of the notional',
  trade_date            DATE      NOT NULL COMMENT 'Date trade was booked',
  maturity_date         DATE      NOT NULL COMMENT 'Date trade matures',
  fx_rate_vs_usd        DOUBLE    NOT NULL COMMENT 'FX rate used to convert notional to USD. USD positions use 1.0',
  notional_usd          DOUBLE    NOT NULL COMMENT 'Notional converted to USD using fx_rate_vs_usd',
  current_price         DOUBLE             COMMENT 'Latest available closing price for the ticker. Null if ticker has no price history',
  latest_price_date     DATE               COMMENT 'Date of the current_price',
  market_value_usd      DOUBLE             COMMENT 'Current market value in USD. Uses notional_usd as proxy where price unavailable',
  days_to_maturity      INT                COMMENT 'Calendar days from today to maturity_date. Negative means already matured',
  direction_multiplier  INT       NOT NULL COMMENT '1 for LONG, -1 for SHORT. Used in net exposure and VaR calculations',
  net_exposure_usd      DOUBLE    NOT NULL COMMENT 'Signed exposure in USD. Positive for LONG, negative for SHORT',
  _silver_loaded_at     TIMESTAMP          COMMENT 'Timestamp when row was loaded into Silver'
)
USING DELTA
COMMENT 'Trading positions enriched with FX rates and latest market prices. All monetary values in USD. One row per trade_id.'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
);

In [0]:
INSERT OVERWRITE market_risk_dev.silver.positions_enriched

WITH

-- Step 1: Extend the FX rates table to include USD explicitly.
--         USD is the base currency so it does not appear in bronze.fx_rates.
--         Adding it here means USD positions join cleanly without needing
--         a COALESCE fallback — cleaner logic, no silent defaults.
fx_with_usd AS (
  SELECT
    UPPER(CAST(currency    AS STRING)) AS currency,
    CAST(rate_vs_usd       AS DOUBLE)  AS rate_vs_usd
  FROM market_risk_dev.bronze.fx_rates
  UNION ALL
  SELECT
    'USD' AS currency,
    1.0   AS rate_vs_usd
),

-- Step 2: Find the most recent price date available for each ticker.
--         We use MAX(price_date) rather than CURRENT_DATE() because
--         the latest data may be 1-2 days old due to market close timing.
latest_price_dates AS (
  SELECT
    ticker,
    MAX(price_date) AS latest_date
  FROM market_risk_dev.silver.prices_cleaned
  GROUP BY ticker
),

-- Step 3: Retrieve the closing price on that latest date.
--         INNER JOIN ensures we only get one row per ticker —
--         the join condition on both ticker AND date is the dedup mechanism.
latest_prices AS (
  SELECT
    pc.ticker,
    CAST(pc.close_price AS DOUBLE) AS current_price,
    pc.price_date                  AS latest_price_date
  FROM market_risk_dev.silver.prices_cleaned  pc
  INNER JOIN latest_price_dates              lpd
    ON  pc.ticker     = lpd.ticker
    AND pc.price_date = lpd.latest_date
),

-- Step 4: Resolve direction multiplier.
--         Centralising this logic in a CTE avoids repeating the CASE
--         expression in multiple downstream columns.
positions_with_direction AS (
  SELECT
    CAST(trade_id     AS STRING)  AS trade_id,
    CAST(desk         AS STRING)  AS desk,
    CAST(counterparty AS STRING)  AS counterparty,
    CAST(asset_class  AS STRING)  AS asset_class,
    CAST(ticker       AS STRING)  AS ticker,
    CAST(direction    AS STRING)  AS direction,
    CAST(notional     AS DOUBLE)  AS notional,
    UPPER(CAST(currency AS STRING)) AS currency,
    CAST(trade_date   AS DATE)    AS trade_date,
    CAST(maturity_date AS DATE)   AS maturity_date,
    CASE direction
      WHEN 'LONG'  THEN  1
      WHEN 'SHORT' THEN -1
      ELSE 0
    END                           AS direction_multiplier
  FROM market_risk_dev.bronze.positions
  WHERE trade_id IS NOT NULL
),

-- Step 5: Final enrichment — join all CTEs together.
--         LEFT JOIN to latest_prices because not all tickers in the
--         trading book have price history (e.g. some synthetic tickers).
--         INNER JOIN to fx_with_usd because every position must have
--         a currency — a missing FX rate indicates a data quality problem
--         that should surface as a failed row, not a silent null.
enriched AS (
  SELECT
    p.trade_id,
    p.desk,
    p.counterparty,
    p.asset_class,
    p.ticker,
    p.direction,
    p.notional,
    p.currency,
    p.trade_date,
    p.maturity_date,
    fx.rate_vs_usd                                          AS fx_rate_vs_usd,
    ROUND(p.notional * fx.rate_vs_usd, 2)                  AS notional_usd,
    lp.current_price,
    lp.latest_price_date,
    -- Market value: use current_price if available, else fall back to notional_usd
    ROUND(
      COALESCE(lp.current_price, p.notional) * fx.rate_vs_usd,
      2
    )                                                       AS market_value_usd,
    DATEDIFF(p.maturity_date, CURRENT_DATE())               AS days_to_maturity,
    p.direction_multiplier,
    ROUND(p.notional * fx.rate_vs_usd * p.direction_multiplier, 2)
                                                            AS net_exposure_usd,
    current_timestamp()                                     AS _silver_loaded_at
  FROM positions_with_direction  p
  INNER JOIN fx_with_usd         fx ON p.currency      = fx.currency
  LEFT JOIN  latest_prices       lp ON p.ticker        = lp.ticker
)

SELECT
  trade_id,
  desk,
  counterparty,
  asset_class,
  ticker,
  direction,
  notional,
  currency,
  trade_date,
  maturity_date,
  fx_rate_vs_usd,
  notional_usd,
  current_price,
  latest_price_date,
  market_value_usd,
  days_to_maturity,
  direction_multiplier,
  net_exposure_usd,
  _silver_loaded_at
FROM enriched;

In [0]:
-- ── Quality gate ──────────────────────────────────────────────────────────────
WITH quality_summary AS (
  SELECT
    COUNT(*)                                              AS total_rows,
    COUNT(DISTINCT trade_id)                              AS unique_trades,
    COUNT(DISTINCT desk)                                  AS unique_desks,
    COUNT(DISTINCT currency)                              AS unique_currencies,
    COUNT(CASE WHEN trade_id         IS NULL THEN 1 END)  AS null_trade_id,
    COUNT(CASE WHEN notional_usd     IS NULL THEN 1 END)  AS null_notional_usd,
    COUNT(CASE WHEN fx_rate_vs_usd   IS NULL THEN 1 END)  AS null_fx_rate,
    COUNT(CASE WHEN net_exposure_usd IS NULL THEN 1 END)  AS null_net_exposure,
    COUNT(*) - COUNT(DISTINCT trade_id)                   AS duplicate_trade_ids,
    COUNT(CASE WHEN notional_usd    <= 0    THEN 1 END)   AS non_positive_notional,
    COUNT(CASE WHEN direction_multiplier NOT IN (1, -1)
               THEN 1 END)                                AS invalid_direction,
    COUNT(CASE WHEN current_price IS NOT NULL THEN 1 END) AS positions_with_price,
    COUNT(CASE WHEN current_price IS NULL     THEN 1 END) AS positions_without_price
  FROM market_risk_dev.silver.positions_enriched
)

SELECT
  total_rows,
  unique_trades,
  unique_desks,
  unique_currencies,
  null_trade_id,
  null_notional_usd,
  null_fx_rate,
  null_net_exposure,
  duplicate_trade_ids,
  non_positive_notional,
  invalid_direction,
  positions_with_price,
  positions_without_price,
  CASE
    WHEN null_trade_id         > 0 THEN 'FAIL — null trade_id found'
    WHEN null_notional_usd     > 0 THEN 'FAIL — null notional_usd found'
    WHEN null_fx_rate          > 0 THEN 'FAIL — null fx_rate found'
    WHEN duplicate_trade_ids   > 0 THEN 'FAIL — duplicate trade_ids found'
    WHEN non_positive_notional > 0 THEN 'FAIL — non-positive notional found'
    WHEN invalid_direction     > 0 THEN 'FAIL — invalid direction multiplier'
    ELSE 'PASS'
  END AS quality_status
FROM quality_summary;

-- ── Desk breakdown ────────────────────────────────────────────────────────────
-- Runs as a second result set in the same cell.
-- Use this to spot anomalies — unexpected desk names, zero positions,
-- or one desk carrying disproportionate exposure.
SELECT
  desk,
  COUNT(*)                        AS position_count,
  COUNT(DISTINCT counterparty)    AS unique_counterparties,
  COUNT(DISTINCT asset_class)     AS unique_asset_classes,
  ROUND(SUM(notional_usd), 2)     AS total_notional_usd,
  ROUND(SUM(net_exposure_usd), 2) AS net_exposure_usd,
  COUNT(CASE WHEN direction = 'LONG'  THEN 1 END) AS long_positions,
  COUNT(CASE WHEN direction = 'SHORT' THEN 1 END) AS short_positions
FROM market_risk_dev.silver.positions_enriched
GROUP BY desk
ORDER BY total_notional_usd DESC;

In [0]:
SELECT
  'prices_cleaned'     AS table_name,
  COUNT(*)             AS row_count,
  COUNT(DISTINCT ticker) AS unique_tickers
FROM market_risk_dev.silver.prices_cleaned

UNION ALL

SELECT
  'positions_enriched' AS table_name,
  COUNT(*)             AS row_count,
  COUNT(DISTINCT desk) AS unique_tickers
FROM market_risk_dev.silver.positions_enriched

ORDER BY table_name;